# Caso 01 — BrightCart (Moda Online): ¿Qué vendió más?
## Día 3 — Del «cuánto» al «qué»: Categorías, Prenda Estrella y Conversión por Canal

En el tercer día bajamos del agregado global al detalle por grupo (`GROUP BY`) para responder a las tres preguntas de **Nuria (Directora de Marketing)** sobre la campaña de **Rebajas de Verano 2024 (`campana_id == 1`)**:
1. **¿Superamos las ventas del año pasado?** Comparativa contra el benchmark histórico (`512.340,00 €`).
2. **¿Qué categorías y qué producto tiraron de la campaña?** Normalización de las 35 variantes de categoría a 8 familias canónicas y aparición del superventas en unidades (**`Vestido Alma`** con `1.016` prendas).
3. **¿Qué canal funcionó mejor?** Contraste entre volumen de tráfico/ingresos (`instagram`) y eficiencia de conversión (`email` y `directo`).

In [1]:
# 1. Carga de datos limpios de data/clean/ y tablas de contexto
import pandas as pd
import numpy as np

lineas = pd.read_csv('data/clean/lineas_limpias.csv')
pedidos = pd.read_csv('data/clean/pedidos_limpios.csv')
productos = pd.read_csv('data/productos.csv')
campana = pd.read_csv('data/campana.csv')
visitas = pd.read_csv('data/visitas_web.csv')

print('Datos cargados exitosamente.')
print(f'Líneas limpias disponibles:  {len(lineas):,}')
print(f'Pedidos limpios disponibles: {len(pedidos):,}')

Datos cargados exitosamente.
Líneas limpias disponibles:  19,986
Pedidos limpios disponibles: 8,000


### Análisis de la Carga de Datos Limpios
* Partimos de las `19.986` líneas válidas y `8.000` pedidos saneados en el Día 2, listos para cruzar con el catálogo de `productos` (`400` referencias), `campana` (`4` campañas) y `visitas_web` (`112` registros diarios).

---
## Reto 1 (Día 3): Construir la Tabla Maestra de la Campaña (`campana_id == 1`) y Comparar contra el Año Anterior
Cruzamos `lineas` con `pedidos` y `productos`, filtramos exclusivamente las **Rebajas de Verano 2024 (`campana_id == 1`)** y contrastamos la facturación bruta pagada en caja frente a las ventas del verano anterior (`512.340,00 €`).

In [2]:
# =============================================================================
# RETO 1 (DÍA 3): CONSTRUIR EL DATAFRAME DE LA CAMPAÑA Y COMPARATIVA YoY
# =============================================================================
df = lineas.merge(pedidos, on='pedido_id', how='inner')
df = df.merge(productos, on='producto_id', how='inner')
df_verano = df[df['campana_id'] == 1].copy()

ingresos_verano_2024 = df_verano['importe_linea'].sum()
ventas_verano_2023 = float(campana.loc[campana['campana_id'] == 1, 'ventas_ano_anterior'].iloc[0])
crecimiento_abs = ingresos_verano_2024 - ventas_verano_2023
crecimiento_pct = (crecimiento_abs / ventas_verano_2023) * 100

print(f'Total pedidos en Rebajas Verano 2024:   {df_verano["pedido_id"].nunique():,}')
print(f'Total líneas de venta en la campaña:    {len(df_verano):,}')
print(f'Total prendas vendidas en la campaña:   {df_verano["cantidad"].sum():,}')
print(f'Ingresos pagados en caja (Verano 2024): {ingresos_verano_2024:,.2f} €')
print(f'Ventas año anterior (Verano 2023):      {ventas_verano_2023:,.2f} €')
print(f'Variación interanual (Bruta en caja):   +{crecimiento_abs:,.2f} € (+{crecimiento_pct:.2f}%)')

Total pedidos en Rebajas Verano 2024:   6,819
Total líneas de venta en la campaña:    17,044
Total prendas vendidas en la campaña:   20,008
Ingresos pagados en caja (Verano 2024): 572,301.82 €
Ventas año anterior (Verano 2023):      512,340.00 €
Variación interanual (Bruta en caja):   +59,961.82 € (+11.70%)


### Análisis del Reto 1 (Volumen de Campaña y Comparativa Interanual)
* La campaña de **Rebajas de Verano 2024** cerró con **`6.819` pedidos válidos**, **`17.044` líneas de venta** y **`20.008` prendas vendidas**, recaudando **`572.301,82 €` en caja**.
* **Límites de los datos ante la petición de Nuria/Carmen:** En `campana.csv` solo existe el total histórico agregado del año anterior (**`512.340,00 €`**), no el desglose por producto de 2023. Aplicando la regla de comunicación ejecutiva (*decir qué sí se puede medir y por qué*), confirmamos que a nivel de facturación cobrada en caja la campaña superó en **`+59.961,82 €` (`+11,70 %`)** la cifra de 2023 (pendiente aún de auditar las devoluciones en el Día 4).

---
## Reto 2 (Día 3): ¿Qué Categorías vendieron más? (La trampa de las grafías sin limpiar)
Agrupamos por `categoria` en crudo para comprobar por qué nunca se debe lanzar un `GROUP BY` sobre una dimensión de texto sin antes auditar sus valores únicos.

In [3]:
# =============================================================================
# RETO 2 (DÍA 3): RANKING DE VENTAS POR CATEGORÍA (SIN NORMALIZAR)
# =============================================================================
cat_ranking = (
    df_verano.groupby('categoria')
    .agg(
        ingresos=('importe_linea', 'sum'),
        unidades=('cantidad', 'sum')
    )
    .sort_values(by='ingresos', ascending=False)
)

print(f'Número de categorías distintas en crudo: {len(cat_ranking)}')
print('Top 10 Categorías en Rebajas de Verano (sin normalizar):')
display(cat_ranking.head(10))

Número de categorías distintas en crudo: 35
Top 10 Categorías en Rebajas de Verano (sin normalizar):


,ingresos,unidades
categoria,,
Vestidos,57310.309,1892
vestido,34723.203,1006
CALZADO,32352.460,678
Zapatos,31743.009,810
calzado,30300.440,667
Calzado,27227.309,568
vestidos,25455.441,814
ABRIGOS,22648.517,460
VESTIDOS,21465.525,697


### Análisis del Reto 2 (Categorías Fragmentadas)
* En el ranking sin limpiar aparecen **`35` categorías distintas**: `'Vestidos'`, `'vestido'`, `'vestidos'` y `'VESTIDOS'` compiten entre sí en filas separadas, igual que `'CALZADO'`, `'Calzado'`, `'calzado'`, `'Zapatos'` y `'zapatos'`. Si entregáramos esta tabla a Dirección, subestimaríamos por completo el peso real de cada línea de negocio.

---
## Reto 3 (Día 3): Normalizar Categorías y Obtener el Verdadero Ranking Comercial
Unificamos las 35 variantes en las **8 categorías canónicas** del catálogo de BrightCart y recalculamos los ingresos, las unidades y la cuota sobre la facturación total de la campaña.

In [4]:
# =============================================================================
# RETO 3 (DÍA 3): NORMALIZAR CATEGORÍAS Y RE-CALCULAR RANKING REAL
# =============================================================================
def unificar_categoria(c):
    c = str(c).strip().lower().replace('é', 'e')
    if 'vestid' in c: return 'Vestidos'
    if 'calzad' in c or 'zapat' in c: return 'Calzado'
    if 'abrig' in c: return 'Abrigos'
    if 'pantalon' in c: return 'Pantalones'
    if 'camiset' in c: return 'Camisetas'
    if 'fald' in c: return 'Faldas'
    if 'jers' in c: return 'Jerséis'
    if 'accesorio' in c or 'complement' in c: return 'Accesorios'
    return c.title()

df_verano['categoria'] = df_verano['categoria'].apply(unificar_categoria)

cat_ranking_limpio = (
    df_verano.groupby('categoria')
    .agg(
        lineas=('linea_id', 'count'),
        unidades=('cantidad', 'sum'),
        ingresos=('importe_linea', 'sum')
    )
    .sort_values(by='ingresos', ascending=False)
)

cat_ranking_limpio['precio_medio_ud'] = cat_ranking_limpio['ingresos'] / cat_ranking_limpio['unidades']
cat_ranking_limpio['pct_ingresos'] = (cat_ranking_limpio['ingresos'] / cat_ranking_limpio['ingresos'].sum()) * 100

print('Ranking REAL y consolidado de las 8 categorías en Rebajas de Verano:')
display(cat_ranking_limpio.style.format({
    'lineas': '{:,}',
    'unidades': '{:,}',
    'ingresos': '{:,.2f} €',
    'precio_medio_ud': '{:,.2f} €',
    'pct_ingresos': '{:.2f}%'
}))

Ranking REAL y consolidado de las 8 categorías en Rebajas de Verano:


,lineas,unidades,ingresos,precio_medio_ud,pct_ingresos
categoria,,,,,
Vestidos,"4,154","4,870","154,292.26 €",31.68 €,26.96%
Calzado,"2,667","3,134","141,720.87 €",45.22 €,24.76%
Pantalones,"1,674","1,965","54,457.69 €",27.71 €,9.52%
Abrigos,833,"1,000","49,503.03 €",49.50 €,8.65%
Camisetas,"2,831","3,330","44,698.04 €",13.42 €,7.81%
Jerséis,"1,374","1,603","44,342.38 €",27.66 €,7.75%
Faldas,"1,463","1,701","41,849.37 €",24.60 €,7.31%
Accesorios,"2,048","2,405","41,438.19 €",17.23 €,7.24%


### Análisis del Reto 3 (El Verdadero Ranking por Categoría)
* Una vez consolidadas las 8 familias, dos categorías concentran **más de la mitad de toda la facturación bruta (`51,72 %`)** de las Rebajas de Verano:
  1. **`Vestidos`:** #1 absoluto con **`154.292,26 €` (`26,96 %` del total)** y **`4.870` unidades** vendidas (`31,68 €/ud`).
  2. **`Calzado`:** #2 con **`141.720,87 €` (`24,76 %` del total)** y **`3.134` unidades** vendidas (`45,22 €/ud`).
* Las seis categorías restantes (`Pantalones` `9,52 %`, `Abrigos` `8,65 %`, `Camisetas` `7,81 %`, `Jerséis` `7,75 %`, `Faldas` `7,31 %` y `Accesorios` `7,24 %`) se reparten de forma muy equilibrada el `48,28 %` restante.

---
## Reto 4 (Día 3): Top 10 Productos Más Vendidos (El fenómeno del «Vestido Alma»)
Bajamos al nivel de referencia (`producto_id` y `nombre`) para identificar qué prenda concreta lideró la tracción de unidades e ingresos durante las rebajas.

In [5]:
# =============================================================================
# RETO 4 (DÍA 3): TOP 10 PRODUCTOS POR VOLUMEN Y FACTURACIÓN
# =============================================================================
top_productos = (
    df_verano.groupby(['producto_id', 'nombre', 'categoria'])
    .agg(
        lineas=('linea_id', 'count'),
        unidades=('cantidad', 'sum'),
        ingresos=('importe_linea', 'sum')
    )
    .sort_values(by='unidades', ascending=False)
)

top_productos['precio_medio_efectivo'] = top_productos['ingresos'] / top_productos['unidades']

print('Top 10 Productos Más Vendidos en Rebajas de Verano (por unidades):')
display(top_productos.head(10).style.format({
    'lineas': '{:,}',
    'unidades': '{:,}',
    'ingresos': '{:,.2f} €',
    'precio_medio_efectivo': '{:,.2f} €'
}))

Top 10 Productos Más Vendidos en Rebajas de Verano (por unidades):


,,,lineas,unidades,ingresos,precio_medio_efectivo
producto_id,nombre,categoria,,,,
1042,Vestido Alma,Vestidos,858,"1,016","30,473.90 €",29.99 €
1204,Top Ola,Camisetas,505,602,"5,761.27 €",9.57 €
1315,Cárdigan Faro,Jerséis,355,420,"10,604.79 €",25.25 €
1362,Sombrero Brisa,Accesorios,279,321,"6,530.53 €",20.34 €
1285,Falda Menta Kaki,Faldas,242,275,"8,477.68 €",30.83 €
1229,Short Menta,Pantalones,219,257,"7,767.23 €",30.22 €
1053,Vestido Laguna Azul,Vestidos,196,241,"7,108.95 €",29.50 €
1106,Alpargata Estío,Calzado,182,215,"12,304.77 €",57.23 €
1252,Short Viento,Pantalones,174,210,"5,575.10 €",26.55 €


### Análisis del Reto 4 (El Pelotazo Aparente del `Vestido Alma`)
* El **`Vestido Alma` (`producto_id = 1042`, categoría `Vestidos`)** arrasa en el ranking comercial:
  * **#1 en unidades:** **`1.016` prendas vendidas** en `858` líneas de pedido, superando en un **`+68,77 %`** al segundo producto más vendido (`Top Ola` con `602` unidades) y más que duplicando al tercero (`Cárdigan Faro` con `420` unidades).
  * **#1 en facturación bruta:** **`30.473,90 €`** cobrados en caja a un precio medio efectivo de **`29,99 €/unidad`** (representando por sí solo el `19,75 %` de toda la categoría `Vestidos` y el `5,32 %` de toda la campaña).
* **Alerta frente al sesgo de confirmación:** Con este dato en la mano, Nuria quiere pedir inmediatamente **`2.000` unidades de reposición para agosto**. Sin embargo, vender muchas unidades no equivale a ganar dinero si aún no hemos cruzado con `devoluciones.csv`.

---
## Reto 5 (Día 3): Rendimiento y Tasa de Conversión por Canal de Adquisición
Evaluamos los 4 canales de captación (`instagram`, `google`, `directo`, `email`) cruzando los pedidos e ingresos de la campaña con las sesiones registradas en `visitas_web.csv` entre el `2024-07-01` y el `2024-07-21`.

In [6]:
# =============================================================================
# RETO 5 (DÍA 3): RENDIMIENTO Y CONVERSIÓN POR CANAL DE ADQUISICIÓN
# =============================================================================
canal_ranking = (
    df_verano.groupby('canal')
    .agg(
        ingresos=('importe_linea', 'sum'),
        pedidos=('pedido_id', 'nunique'),
        unidades=('cantidad', 'sum')
    )
    .sort_values(by='ingresos', ascending=False)
)

# Cruce con sesiones web durante la campaña (2024-07-01 a 2024-07-21)
visitas['fecha'] = pd.to_datetime(visitas['fecha'])
vis_verano = (
    visitas[(visitas['fecha'] >= '2024-07-01') & (visitas['fecha'] <= '2024-07-21')]
    .groupby('canal')['sesiones']
    .sum()
)

canal_ranking['sesiones_web'] = canal_ranking.index.map(vis_verano)
canal_ranking['conversion_pct'] = (canal_ranking['pedidos'] / canal_ranking['sesiones_web']) * 100
canal_ranking['ticket_medio'] = canal_ranking['ingresos'] / canal_ranking['pedidos']
canal_ranking['prendas_por_pedido'] = canal_ranking['unidades'] / canal_ranking['pedidos']
canal_ranking['pct_ingresos'] = (canal_ranking['ingresos'] / canal_ranking['ingresos'].sum()) * 100

print('Rendimiento y Conversión por Canal en Rebajas de Verano (01/07 a 21/07):')
display(canal_ranking.style.format({
    'ingresos': '{:,.2f} €',
    'pedidos': '{:,}',
    'unidades': '{:,}',
    'sesiones_web': '{:,.0f}',
    'conversion_pct': '{:.2f}%',
    'ticket_medio': '{:,.2f} €',
    'prendas_por_pedido': '{:.2f}',
    'pct_ingresos': '{:.2f}%'
}))

Rendimiento y Conversión por Canal en Rebajas de Verano (01/07 a 21/07):


,ingresos,pedidos,unidades,sesiones_web,conversion_pct,ticket_medio,prendas_por_pedido,pct_ingresos
canal,,,,,,,,
instagram,"201,623.15 €","2,400","7,096","57,624",4.16%,84.01 €,2.96,35.23%
google,"142,900.62 €","1,714","5,001","34,923",4.91%,83.37 €,2.92,24.97%
directo,"114,512.39 €","1,344","3,949","17,096",7.86%,85.20 €,2.94,20.01%
email,"113,265.66 €","1,361","3,962","12,337",11.03%,83.22 €,2.91,19.79%


### Análisis del Reto 5 (Volumen de Tráfico vs. Eficiencia de Conversión)
* **No es lo mismo traer visitas que convertir pedidos:**
  * **`instagram`** lidera en volumen bruto gracias a los `48.000 €` de presupuesto publicitario: atrae **`57.624` sesiones (`47,2 %` del tráfico)** y genera **`2.400` pedidos** y **`201.623,15 €` (`35,23 %` de los ingresos brutos)**, lo que supone un **ROAS bruto aparente de `4,20x`**. Sin embargo, tiene la **tasa de conversión más baja (`4,16 %`)**, propia del tráfico frío de redes sociales.
  * **`email`** es el polo opuesto: con apenas **`12.337` sesiones**, logra **`1.361` pedidos** y **`113.265,66 €`** gracias a una **tasa de conversión del `11,03 %` (2,65 veces superior a Instagram)** y coste de pauta cero, al dirigirse a base de clientes propios.
  * **`directo`** también muestra alta intención de compra (**`7,86 %` de conversión** y el **ticket medio más alto, `85,20 €`**), mientras que **`google`** aporta el `24,97 %` de los ingresos (`142.900,62 €`) con un `4,91 %` de conversión.

---
## Conclusión Ejecutiva del Día 3
1. **En la superficie, todo parece un éxito rotundo:** La campaña facturó **`572.301,82 €` en caja** (`+11,70 %` vs los `512.340,00 €` de 2023), liderada por **`Vestidos` (`154.292,26 €`)** y **`Calzado` (`141.720,87 €`)**, con el **`Vestido Alma` (`1.016` uds, `30.473,90 €`)** como superventas indiscutible e **`Instagram Ads`** mostrando un ROAS bruto de `4,20x`.
2. **Por qué hemos frenado el pedido de 2.000 unidades de Nuria:** Todas las métricas vistas hoy miden lo que **salió del almacén**, pero ignoran lo que **volvió**. Mañana (Día 4) cruzaremos este cuadro de mando con `devoluciones.csv` para descubrir cuánto dinero quedó realmente en BrightCart.